In [ ]:
rundir = '/gpfs/commons/home/tbotella/bam-readcount-rs/bench/results/latest'

In [ ]:
from pathlib import Path
import polars as pl
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'svg'
mpl.rcParams['figure.dpi'] = 110
mpl.rcParams['savefig.dpi'] = 200
mpl.rcParams['savefig.bbox'] = 'tight'
mpl.rcParams['font.family'] = 'DejaVu Sans'
rundir = Path(rundir)
plots = rundir / 'plots'
plots.mkdir(parents=True, exist_ok=True)
joined_dir = rundir / 'joined'
print('rundir:', rundir, '\nplots:', plots, '\nparquets:', sorted(joined_dir.iterdir())[:3])

In [ ]:
INFO = [
    'count','avg_mapping_quality','avg_basequality','avg_se_mapping_quality',
    'num_plus_strand','num_minus_strand','avg_pos_as_fraction',
    'avg_num_mismatches_as_fraction','avg_sum_mismatch_qualities',
    'num_q2_containing_reads','avg_distance_to_q2_start_in_q2_reads',
    'avg_clipped_length','avg_distance_to_effective_3p_end',
]
lf = pl.scan_parquet(str(joined_dir / '*.parquet'))
n_rows = lf.select(pl.len()).collect().item()
n_samples = sum(1 for _ in joined_dir.glob('*.parquet'))
print(f'Lazy scan over {n_samples} samples, {n_rows:,} joined rows.')

In [ ]:
# SUBSAMPLE first — read each per-sample parquet, take 50 rows. ~98k rows total.
# Then compute per-feature stats on the subsample. Pearson r from 95k uniform-ish
# samples is statistically equivalent to from 2.4B rows (sigma_r ~ 1/sqrt(n) ≈ 0.003).
import concurrent.futures

PER_FILE = 50

def _sample_one(p):
    df = pl.read_parquet(p)
    if df.height > PER_FILE:
        return df.sample(n=PER_FILE, seed=42)
    return df

parquet_files = sorted(joined_dir.glob('*.parquet'))
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    parts = list(ex.map(_sample_one, parquet_files))
sample_df = pl.concat(parts)
print(f'subsample: {sample_df.shape[0]:,} rows from {len(parquet_files)} parquets')

rows = []
n_sub = sample_df.shape[0]
for m in INFO:
    a_col, b_col = f'ref_{m}', f'rs_{m}'
    a = sample_df[a_col].to_numpy()
    b = sample_df[b_col].to_numpy()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if len(a) < 2 or a.std() == 0 or b.std() == 0:
        r = 1.0
    else:
        r = float(np.corrcoef(a, b)[0, 1])
    diff = np.abs(a - b)
    mae = float(diff.mean()) if len(a) else 0.0
    pct_within = float((diff <= 0.01).mean() * 100.0) if len(a) else 0.0
    pct_exact = float((a == b).mean() * 100.0) if len(a) else 0.0
    rows.append({
        'metric': m,
        'n': len(a),
        'pearson_r': r,
        'mae': mae,
        'pct_within_001': pct_within,
        'pct_exact': pct_exact,
    })
corr_df = pl.DataFrame(rows)
corr_df.write_csv(rundir / 'per_feature_corr.tsv', separator='\t')
print(corr_df)


In [ ]:
# UNIFIED CORRELATION — MAIN figure. All 13 metrics overlaid on a single
# [0,1]x[0,1] axes, each metric per-metric normalized. Uses subsample.
fig, ax = plt.subplots(figsize=(8.5, 8.5))
cmap = plt.get_cmap('tab20')
rng = np.random.default_rng(0)
n_per_metric = 5000

def fmt_n(n):
    if n >= 1e9: return f'{n/1e9:.2f}B'
    if n >= 1e6: return f'{n/1e6:.1f}M'
    if n >= 1e3: return f'{n/1e3:.0f}k'
    return str(n)

for i, m in enumerate(INFO):
    a = sample_df[f'ref_{m}'].to_numpy()
    b = sample_df[f'rs_{m}'].to_numpy()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if len(a) == 0:
        continue
    if len(a) > n_per_metric:
        idx = rng.choice(len(a), size=n_per_metric, replace=False)
        a, b = a[idx], b[idx]
    lo = float(a.min())
    hi = float(a.max())
    if hi - lo < 1e-9:
        hi = lo + 1.0
    a_n = (a - lo) / (hi - lo)
    b_n = (b - lo) / (hi - lo)
    r = corr_df.filter(pl.col('metric') == m)['pearson_r'][0]
    ax.scatter(a_n, b_n, s=5, alpha=0.45, edgecolor='none',
               color=cmap(i % 20), label=f'{m}  (r={r:.4f})')
ax.plot([0, 1], [0, 1], color='black', lw=1.2, ls='--', zorder=10, label='y = x')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_xlabel('upstream bam-readcount (per-metric normalized)')
ax.set_ylabel('bam-readcount-rs (per-metric normalized)')
ax.set_title(
    f'bam-readcount-rs vs upstream — all 13 metrics\n'
    f'{fmt_n(n_rows)} joined rows × {n_samples} samples (sub={fmt_n(sample_df.shape[0])})'
)
ax.legend(fontsize=7.5, loc='lower right', frameon=True, framealpha=0.92)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2, lw=0.5)
fig.tight_layout()
fig.savefig(plots / 'correlation_unified.png', dpi=200)
plt.show()


In [ ]:
# PER-METRIC 2D HISTOGRAM GRID — density rather than scatter (better for a
# 2.4B-row dataset; scatter saturates anyway). Uses the in-memory subsample.
import math
ncols = 4
nrows = math.ceil(len(INFO) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.0*ncols, 3.6*nrows))
axes = axes.flatten()
panel_letters = list('ABCDEFGHIJKLMN')
for i, m in enumerate(INFO):
    ax = axes[i]
    a = sample_df[f'ref_{m}'].to_numpy()
    b = sample_df[f'rs_{m}'].to_numpy()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    r = corr_df.filter(pl.col('metric') == m)['pearson_r'][0]
    pct = corr_df.filter(pl.col('metric') == m)['pct_exact'][0]
    if len(a) > 0:
        ax.hist2d(a, b, bins=120, cmap='magma', cmin=1)
        lo = float(min(a.min(), b.min()))
        hi = float(max(a.max(), b.max()))
        ax.plot([lo, hi], [lo, hi], color='cyan', lw=0.8, ls='--', zorder=10)
    ax.set_xlabel(f'upstream {m}', fontsize=8)
    ax.set_ylabel(f'rs {m}', fontsize=8)
    ax.tick_params(labelsize=7)
    color = '#0a8a3a' if r >= 0.99 else ('#c4a000' if r >= 0.95 else '#a30000')
    ax.set_title(f'{panel_letters[i]}  {m}\nr={r:.4f}  exact={pct:.1f}%', fontsize=8.5, color=color)
for j in range(len(INFO), len(axes)):
    axes[j].axis('off')
fig.suptitle(
    f'bam-readcount-rs vs upstream — 2D density per metric '
    f'({fmt_n(n_rows)} rows / {n_samples} samples, sub={fmt_n(sample_df.shape[0])})',
    fontsize=11, y=1.005,
)
fig.tight_layout()
fig.savefig(plots / 'correlation_grid.png')
plt.show()


In [ ]:
# RUNTIME / MEMORY (bam-readcount-rs only — upstream timings would come
# from Nextflow trace files, see README).
ms = pl.read_csv(rundir / 'per_sample_metrics.tsv', separator='\t').with_columns([
    pl.col('rs_wall_s').cast(pl.Float64),
    pl.col('rs_peak_rss_kb').cast(pl.Float64),
    pl.col('bed_lines').cast(pl.Int64),
])
print(ms.describe())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
for c in ms['cohort'].unique().to_list():
    sub = ms.filter(pl.col('cohort') == c)
    ax.scatter(sub['bed_lines'], sub['rs_wall_s'], label=c, s=14, alpha=0.7)
ax.set_xlabel('queried sites (BED lines)')
ax.set_ylabel('wall time (s, 4 threads)')
ax.set_title(f'Runtime — median {ms["rs_wall_s"].median():.1f}s across {len(ms)} samples')
ax.legend(fontsize=8, frameon=False)
ax.set_xscale('log'); ax.set_yscale('log')
ax = axes[1]
for c in ms['cohort'].unique().to_list():
    sub = ms.filter(pl.col('cohort') == c)
    ax.scatter(sub['bed_lines'], sub['rs_peak_rss_kb'] / 1024.0, label=c, s=14, alpha=0.7)
ax.set_xlabel('queried sites (BED lines)')
ax.set_ylabel('peak RSS (MB)')
ax.set_title(f'Memory — median {ms["rs_peak_rss_kb"].median()/1024:.0f} MB')
ax.legend(fontsize=8, frameon=False)
ax.set_xscale('log')
fig.tight_layout()
fig.savefig(plots / 'runtime_memory.png')
plt.show()

In [ ]:
# SUMMARY.md — picked up by README
lines = [
    f'# Benchmark summary — `{rundir.name}`',
    '',
    f'- samples scored: **{n_samples}**',
    f'- joined rows (per-sample × position × base): **{n_rows:,}**',
    f'- median rs wall time: **{ms["rs_wall_s"].median():.1f}s** (4 threads)',
    f'- median rs peak RSS: **{ms["rs_peak_rss_kb"].median()/1024:.0f} MB**',
    '',
    '## Per-feature correlation',
    '',
    '| metric | n | pearson r | MAE | % exact | % within 0.01 |',
    '|---|---:|---:|---:|---:|---:|',
]
for r in corr_df.iter_rows(named=True):
    lines.append(f"| {r['metric']} | {r['n']:,} | {r['pearson_r']:.5f} | {r['mae']:.4g} | {r['pct_exact']:.4f}% | {r['pct_within_001']:.4f}% |")
(rundir / 'SUMMARY.md').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))